In [ ]:
#shape variables

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
# Load your CSV
df = pd.read_csv("basic_shape_features.csv")

# Separate metadata and features
non_features = df[['filename', 'country']]
features = df.drop(columns= non_features)

# Standardize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

features_scaled = pd.DataFrame(features_scaled, columns=features.columns)


In [3]:
#shape variables
from scipy.spatial.distance import mahalanobis
from numpy.linalg import inv

X = features_scaled.values
cov_matrix = np.cov(X, rowvar=False)
inv_cov = inv(cov_matrix)
mean_distr = X.mean(axis=0)

distances = []

for i in range(len(X)):
    dist = mahalanobis(X[i], mean_distr, inv_cov)
    distances.append(dist)

df['mahal_dist'] = distances

# Chi-square threshold
from scipy.stats import chi2
threshold = chi2.ppf((1 - 0.001), df=X.shape[1])

df_clean = df[df['mahal_dist'] < np.sqrt(threshold)]

In [ ]:
df_clean['log_area'] = np.log(df_clean['area_px'])
df_clean['log_perimeter'] = np.log(df_clean['perimeter_px'])
df_clean['log_major_axis'] = np.log(df_clean['major_axis_px'])
df_clean['log_minor_axis'] = np.log(df_clean['minor_axis_px'])

log_cols = [
    'log_area', 'log_perimeter', 'log_major_axis', 'log_minor_axis',
    'aspect_ratio', 'extent', 'roundness',
    'circularity', 'convexity', 'solidity']

#creating shape_df and labels variables
shape_df = df_clean[log_cols]
labels = df_clean["country"].values

In [26]:
#efd variables
import os
import numpy as np
import pandas as pd

OUTLINE_ROOT = "all_outlines_cleaned_434"

outlines = []
countries = []
leaf_ids = []

for fname in sorted(os.listdir(OUTLINE_ROOT)):
    if not fname.endswith(".npy"):
        continue
    
    path = os.path.join(OUTLINE_ROOT, fname)
    contour = np.load(path)
    
    # Extract country from filename
    country = fname.split("_")[0]
    
    outlines.append(contour)
    countries.append(country)
    leaf_ids.append(fname)

In [27]:
from pyefd import elliptic_fourier_descriptors
harmonics = 20

efd_features = []

for contour in outlines:
    
    contour = np.array(contour, dtype=np.float64)
    
    coeffs = elliptic_fourier_descriptors(
        contour,
        order=harmonics,
        normalize=True
    )
    
    efd_features.append(coeffs.flatten())

efd_df = pd.DataFrame(efd_features, index = leaf_ids)

In [ ]:
efd_df_clean = efd_df.drop(columns=[0,1,2,3])

print("After removing first harmonic:", efd_df_clean.shape)
efd_df_clean["country"] = countries
efd_df_clean = efd_df_clean.reset_index(drop=True)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Separate features
X = efd_df_clean.drop(columns=["country"])

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca_efd = PCA()
X_pca = pca_efd.fit_transform(X_scaled)


efd_pca_df = pd.DataFrame(
    X_pca,
    index=efd_df_clean.index
)

In [35]:
efd_pca_df.index = (
    efd_pca_df.index
    .str.replace(".npy", "", regex=False)
)

In [36]:
df_clean.index = (
    df_clean.index
    .str.replace(".png", "", regex=False)
)

In [38]:
common_ids = df_clean.index.intersection(efd_pca_df.index)

shape_df = df_clean.loc[common_ids].drop(columns=["country"])
efd_scores = efd_pca_df.loc[common_ids].values
labels = df_clean.loc[common_ids]["country"].values

In [ ]:
#creating efd_scores variable
efd_scores = X_pca
print(efd_scores.shape)

In [ ]:
#reduce efd dimensionality
efd_scores_reduced = efd_pca_df.iloc[:, :20].loc[shape_df.index].values
print(efd_scores_reduced.shape)

In [45]:
X_shape = shape_df.values
X_efd = efd_scores_reduced
X_combined = np.hstack([X_shape, X_efd])

y = labels

In [46]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [47]:
log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000))
])

In [48]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

In [49]:
#step 4
def evaluate_model(X, y, model, name="Model"):
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=cv)
    
    print("\n==============================")
    print(name)
    print("==============================")
    print("CV Accuracy Mean:", round(cv_scores.mean(), 4))
    print("CV Accuracy Std:", round(cv_scores.std(), 4))
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print("\nTest Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [ ]:
#step 5
# Logistic Regression
evaluate_model(X_shape, y, log_model, "LogReg - Shape")
evaluate_model(X_efd, y, log_model, "LogReg - EFD")
evaluate_model(X_combined, y, log_model, "LogReg - Combined")

# Random Forest
evaluate_model(X_shape, y, rf_model, "RF - Shape")
evaluate_model(X_efd, y, rf_model, "RF - EFD")
evaluate_model(X_combined, y, rf_model, "RF - Combined")